# Assignment: Extend the az.ipynb Lab

**Based on:** `Lab2.ipynb` (the Module 3 lab).

This week's assignment is short on purpose: take your working `Lab2.ipynb` lab and add **one
more step** to the chain. No new concepts, no new setup, no new libraries — just one more
chained LLM call that builds on what you already have.

**Two deployments this time:** `gpt-5.1-ptu` is the default deployment for every existing
step (Steps 1–4). The new step you add (Step 5) must call `gpt-5.4-ptu` instead.

## Step 1 — Start from your working lab

- Make a copy of your completed `Lab2.ipynb` (e.g. rename the copy `assignment3.ipynb`), or
  continue directly inside this notebook — either is fine.
- Copy in your working code from the lab's Steps 1–4: the imports and `.env` config, the
  `AzureOpenAI` client, the `chat()` helper, and the chain itself (fun fact → generate a hard
  question → answer it → evaluate the answer).
- Confirm your `.env`'s `AZURE_APIM_OPENAI_DEPLOYMENT` is set to `gpt-5.1-ptu` — this stays
  the default deployment for Steps 1–4, unchanged.

Run those cells first and confirm they still work before moving on.

In [9]:
from dotenv import load_dotenv
import os
import sys

from openai import AzureOpenAI

# Read .env and override any existing process env values.
load_dotenv(override=True)

# APIM settings from .env. Endpoint must be the host only, e.g.
# https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net  (no /openai/deployments)
api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_APIM_OPENAI_API_VERSION")
endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_APIM_OPENAI_DEPLOYMENT")

if not all([api_key, api_version, endpoint, deployment]):
    sys.exit(
        "Missing Azure APIM settings. Set AZURE_APIM_OPENAI_SUBSCRIPTION_KEY, "
        "AZURE_APIM_OPENAI_API_VERSION, AZURE_APIM_OPENAI_ENDPOINT, and "
        "AZURE_APIM_OPENAI_DEPLOYMENT in your .env file."
    )

print(f"Azure APIM key exists and begins {api_key[:8]}")
print(f"Deployment: {deployment}")

# Sync client pointed at Azure APIM.
openai = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

# On Azure the `model` argument is the *deployment name*, not an OpenAI model id.
# GPT-5 deployments need max_completion_tokens (max_tokens is rejected).
def chat(messages, max_completion_tokens=1000, deployment=deployment):
    return openai.chat.completions.create(
        model=deployment,
        messages=messages,
        max_completion_tokens=max_completion_tokens,
    )

# 1) Fun fact
messages = [{"role": "user", "content": "Tell me a short fun fact"}]
response = chat(messages)
print(response.choices[0].message.content)

# 2) Ask the model to invent a hard IQ-style question
question = (
    "Please propose a hard, challenging question to assess someone's IQ. "
    "Respond only with the question."
)
messages = [{"role": "user", "content": question}]
response = chat(messages)
question = response.choices[0].message.content
print(question)

# 3) Ask the model to answer that question
messages = [{"role": "user", "content": question}]
response = chat(messages)
answer = response.choices[0].message.content
print(answer)

# Render the answer as Markdown in the notebook.
from IPython.display import Markdown, display

display(Markdown(answer))

# 4) Ask the model to evaluate the answer
message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""
print(message)

messages = [{"role": "user", "content": message}]
response = chat(messages, max_completion_tokens=5000)
print(response.choices[0].message.content)

Azure APIM key exists and begins f9fd3013
Deployment: gpt-5.1-ptu
Octopuses have three hearts and blue blood—two hearts pump blood to the gills, and one pumps it to the rest of the body.
In a sealed room there is a perfectly frictionless, infinitely long, straight track. On this track are three identical, perfectly elastic spherical balls A, B, and C, all of equal mass. Initially, ball B is at rest exactly midway between A and C; balls A and C are moving toward B with equal speeds v but in opposite directions (A moves right, C moves left). All collisions are perfectly elastic, and there are no external forces, friction, or energy losses of any kind. After a very long time, when no further collisions occur, what is the final speed and direction of each ball (A, B, and C)? Explain your reasoning in detail.
Each ball ends with speed \(v\), moving in the same direction it started, far away from the others. Concretely:

- Ball A: final speed \(v\), moving to the right  
- Ball B: final spee

Each ball ends with speed \(v\), moving in the same direction it started, far away from the others. Concretely:

- Ball A: final speed \(v\), moving to the right  
- Ball B: final speed \(v\), moving to the right  
- Ball C: final speed \(v\), moving to the left  

No ball ever comes to rest permanently. Here is why.

---

### 1. Key facts about 1D elastic collisions of equal masses

For two equal-mass balls colliding elastically in one dimension, their velocities are simply exchanged.

If masses \(m_1 = m_2\) and initial velocities are \(u_1, u_2\), then after an elastic collision:
\[
v_1 = u_2,\quad v_2 = u_1
\]
This follows from conservation of momentum and kinetic energy.

Thus, in our problem, any time two of the balls collide, they just swap velocities.

---

### 2. Initial setup (in the lab frame)

- Ball A: velocity \(v_A = +v\) (to the right)  
- Ball B: velocity \(v_B = 0\)  
- Ball C: velocity \(v_C = -v\) (to the left)  

Ball B is midway between A and C. Because A and C are identical and approach symmetrically, B will be hit by both at the same time.

At that instant, A and C reach B simultaneously and make contact with B from opposite sides.

---

### 3. View in the center-of-mass (COM) frame

This problem is easiest in the COM frame.

Total momentum in the lab frame:
\[
P = m v_A + m v_B + m v_C = m v + m\cdot 0 + m(-v) = 0
\]
So the center of mass is at rest. The lab frame is already the COM frame.

In the COM frame at all times:
- Total momentum is zero.
- Total kinetic energy is fixed:
  \[
  K = \tfrac{1}{2}m v^2 + 0 + \tfrac{1}{2}m v^2 = m v^2
  \]

Because the problem is perfectly symmetric (A and C identical, positions symmetric about B, speeds equal and opposite), the COM stays fixed, and the configuration evolves symmetrically in time.

---

### 4. What happens at the simultaneous collision?

Just before contact:

- A has \(+v\), B has \(0\), C has \(-v\).

A and C reach B together. At that instant, B is squeezed symmetrically between two identical balls approaching with equal and opposite velocities. There is no preferred direction for B to move after the collision due to symmetry.

By symmetry in the COM frame:

- B cannot acquire a net velocity either left or right; the only symmetric option is that B stays at rest (velocity 0) after this instant in the COM frame.
- The total momentum must remain 0, so A and C must leave with equal and opposite velocities.
- Total kinetic energy must remain \(m v^2\).

Let A and C have final speeds \(u\) and \(-u\) after the collision, with B at rest.

Conservation of kinetic energy:
\[
\text{Initial }K = m v^2 = \tfrac{1}{2}m v^2 + 0 + \tfrac{1}{2}m v^2
\]
\[
\text{Final }K = \tfrac{1}{2}m u^2 + 0 + \tfrac{1}{2}m u^2 = m u^2
\]
So:
\[
m v^2 = m u^2 \Rightarrow u = v
\]

Therefore, immediately after the symmetric three-body collision:

- \(v_A = +v\) (right)
- \(v_B = 0\) (still at rest)
- \(v_C = -v\) (left)

So effectively, the situation after the “instant” of interaction is exactly the same as initially, except that A and C have already passed through the point where B sits.

---

### 5. Are there any further collisions?

After that:

- Ball A moves right with speed \(v\), away from B.
- Ball B is at rest at the midpoint.
- Ball C moves left with speed \(v\), away from B.

All balls are now separating: distances between any pair increase with time. Hence no further collisions can occur.

---

### 6. Final state “after a very long time”

As \(t \to \infty\):

- Ball


Here is a question:
In a sealed room there is a perfectly frictionless, infinitely long, straight track. On this track are three identical, perfectly elastic spherical balls A, B, and C, all of equal mass. Initially, ball B is at rest exactly midway between A and C; balls A and C are moving toward B with equal speeds v but in opposite directions (A moves right, C moves left). All collisions are perfectly elastic, and there are no external forces, friction, or energy losses of any kind. After a very long time, when no further collisions occur, what is the final speed and direction of each ball (A, B, and C)? Explain your reasoning in detail.

And here is a possible answer that might be correct or incorrect:
Each ball ends with speed \(v\), moving in the same direction it started, far away from the others. Concretely:

- Ball A: final speed \(v\), moving to the right  
- Ball B: final speed \(v\), moving to the right  
- Ball C: final speed \(v\), moving to the left  

No ball ever come

## Step 2 — Add one more chained step

Add a **5th step** to the chain. Its prompt must be built from at least one variable you
already have (`fact`, `question`, `answer`, or the evaluation text) — the same chaining
pattern as every other step in the lab.

**This step must call `gpt-5.4-ptu`, not the default deployment.** Pass it explicitly when
you call `chat()`:

```python
response = chat(messages, deployment="gpt-5.4-ptu")
```

Pick **one** idea below, or invent your own:

- Rate the difficulty of the question on a 1–10 scale, with a one-sentence justification.
- Rewrite the answer in one simple sentence a 10-year-old could understand.
- Suggest one new, related fun fact that connects to the original topic.
- Translate the final answer into a language of your choice.
- Write a one-line verdict on whether the model's own answer was actually correct, and why.

Store the result in its own variable, and print it clearly labeled (e.g. `=== STEP 5
(gpt-5.4-ptu) ===`).

In [10]:
# 5) Rate the difficulty of the question (gpt-5.4-ptu)
rating_prompt = f"""
Here is an IQ-style question:
{question}

And the answer that was given:
{answer}

Rate the difficulty of this question on a 1-10 scale, with a one-sentence justification.
"""
messages = [{"role": "user", "content": rating_prompt}]
response = chat(messages, deployment="gpt-5.4-ptu")
difficulty_rating = response.choices[0].message.content
print("=== STEP 5 (gpt-5.4-ptu) ===")
print(difficulty_rating)

=== STEP 5 (gpt-5.4-ptu) ===
4/10 — It looks tricky because of the simultaneous three-ball collision, but symmetry plus conservation laws make the correct outcome fairly straightforward once you recognize the lab frame is the center-of-mass frame.


## Reflection

Answer in a sentence or two each:

1. **Which earlier variable(s) did your Step 5 prompt use, and why that one?**
2. **What would break if you ran Step 5 before the step it depends on?**
3. **Why might a real project deliberately use a different deployment (e.g. a stronger or
   more expensive model) for just one step in a chain, instead of using it everywhere?**

### My reflection

1. Difficulty is judged from the question itself, and the answer shows how much reasoning it actually took.
2. The f-string needs question and answer, which only exist after Steps 2–3 run. You genuinely hit the cousin of this today (the stale chat() TypeError), so you can speak to it honestly: cells depend on prior cells' state, and order of execution is load-bearing, not cosmetic.
3. Cost/latency vs. quality routing — the cheap default handles routine steps; you pay for the stronger model only where judgment quality matters. Everywhere would multiply cost for zero gain on the easy steps.

## Submission checklist

- [ ] Notebook runs top to bottom without errors (`Kernel → Restart & Run All`)
- [ ] `.env` file is **not** included in your submission
- [ ] Step 5 is clearly labeled and its prompt uses at least one earlier variable
- [ ] Reflection questions are answered